In [ ]:
import numpy as np
import scipy.io
import mne
from sklearn.preprocessing import MinMaxScaler
import os
from tqdm import tqdm  # Importing tqdm for the progress bar


mne.set_log_level('ERROR')


In [ ]:


def load_hs_mat(file_path, selected_channels):
    """
    Load EEG and kinematics data from a .mat file.
    """
    data = scipy.io.loadmat(file_path)
    hs = data["hs"][0, 0]
    eeg_data = hs["eeg"][0, 0]["sig"]  
    channel_names = [ch[0] for ch in hs["eeg"][0, 0]["names"][0]]
    #kin_columns = [3,7,11] this is the indices of the angles of the wrist

    kin_columns = [19, 20, 21, 23, 24, 25, 27, 28, 29] #this is the indices of index finger, thumb, and wrist x,y,z positions
    kin_data = hs["kin"][0, 0]["sig"][:, kin_columns]
    
    # Select only required EEG channels
    channel_indices = [i for i, ch in enumerate(channel_names) if ch in selected_channels]
    eeg_data = eeg_data[:, channel_indices]
    
    return eeg_data, kin_data, selected_channels


def load_all_data(folder_path, selected_channels):
    """
    Load all EEG and kinematic data from dataset folder.
    """
    eeg_all, kin_all = [], []
    for p in range(1, 13):
        for s in range(1, 10):
            file_name = f"HS_P{p}_S{s}.mat"
            file_path = os.path.join(folder_path, file_name)
            if os.path.exists(file_path):
                eeg_data, kin_data, _ = load_hs_mat(file_path, selected_channels)
                eeg_all.append(eeg_data)
                kin_all.append(kin_data)
                print(f"Loaded: {file_name}")
            else:
                print(f"Missing: {file_name}")
    return eeg_all, kin_all


selected_channels = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "FC5", "FC1", "FC2", "FC6", "T7", "C3", "Cz", "C4", "T8",
    "TP9", "CP5", "CP1", "CP2", "CP6", "TP10", "P7", "P3", "Pz", "P4", "P8",
    "PO9", "O1", "Oz", "O2", "PO10"]


# Load data
folder_path = "./dataset"  # Update path

eeg_data, kin_data = load_all_data(folder_path, selected_channels)
# Save EEG data and Kinematics data to .npy files
eeg_data_path = "eeg_data.npy"
kin_data_path = "kin_data.npy"

# Convert to NumPy arrays and save
np.save(eeg_data_path, np.array(eeg_data))
# np.save(kin_data_path, np.array(kin_data))

print(f"Saved EEG data to {eeg_data_path}")
print(f"Saved Kinematics data to {kin_data_path}")



Loaded: HS_P1_S1.mat
Loaded: HS_P1_S2.mat
Loaded: HS_P1_S3.mat
Loaded: HS_P1_S4.mat
Loaded: HS_P1_S5.mat
Loaded: HS_P1_S6.mat
Loaded: HS_P1_S7.mat
Loaded: HS_P1_S8.mat
Loaded: HS_P1_S9.mat
Loaded: HS_P2_S1.mat
Loaded: HS_P2_S2.mat
Loaded: HS_P2_S3.mat
Loaded: HS_P2_S4.mat
Loaded: HS_P2_S5.mat
Loaded: HS_P2_S6.mat
Loaded: HS_P2_S7.mat
Loaded: HS_P2_S8.mat
Loaded: HS_P2_S9.mat
Loaded: HS_P3_S1.mat
Loaded: HS_P3_S2.mat
Loaded: HS_P3_S3.mat
Loaded: HS_P3_S4.mat
Loaded: HS_P3_S5.mat
Loaded: HS_P3_S6.mat
Loaded: HS_P3_S7.mat
Loaded: HS_P3_S8.mat
Loaded: HS_P3_S9.mat
Loaded: HS_P4_S1.mat
Loaded: HS_P4_S2.mat
Loaded: HS_P4_S3.mat
Loaded: HS_P4_S4.mat
Loaded: HS_P4_S5.mat
Loaded: HS_P4_S6.mat
Loaded: HS_P4_S7.mat
Loaded: HS_P4_S8.mat
Loaded: HS_P4_S9.mat
Loaded: HS_P5_S1.mat
Loaded: HS_P5_S2.mat
Loaded: HS_P5_S3.mat
Loaded: HS_P5_S4.mat
Loaded: HS_P5_S5.mat
Loaded: HS_P5_S6.mat
Loaded: HS_P5_S7.mat
Loaded: HS_P5_S8.mat
Loaded: HS_P5_S9.mat
Loaded: HS_P6_S1.mat
Loaded: HS_P6_S2.mat
Loaded: HS_P6

C:\Users\bigdipper\AppData\Local\Temp\ipykernel_21844\2048872744.py:54: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  np.save(eeg_data_path, np.array(eeg_data))


Saved EEG data to eeg_data.npy
Saved Kinematics data to kin_data.npy


In [ ]:
eeg_data = np.load("eeg_data.npy", allow_pickle=True)
session_lengths = [session.shape[0] for session in eeg_data]
split_indices = np.cumsum(session_lengths[:-1])-1  


In [ ]:

fs=500
num_participants = 12
num_sessions = 108
num_channels = 32  

# Define 32 EEG channels
selected_channels = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
                     "FC5", "FC1", "FC2", "FC6", "T7", "C3", "Cz", "C4", "T8",
                     "TP9", "CP5", "CP1", "CP2", "CP6", "TP10", "P7", "P3", "Pz",
                     "P4", "P8", "PO9", "O1", "Oz", "O2", "PO10"]

mne.set_log_level('WARNING')

# Suppress MNE logging for cleaner output
eeg_data_all=np.load("eeg_data.npy",allow_pickle=True)

# Create MNE Info object
info = mne.create_info(ch_names=selected_channels, sfreq=fs, ch_types="eeg")
arr=[]
for eeg_data in eeg_data_all:

    raw_full = mne.io.RawArray(np.vstack(eeg_data).T, info)
    raw_full.filter(l_freq=0.1, h_freq=40, fir_design="firwin")
    raw_full.set_eeg_reference(ref_channels="average", projection=False)
    arr.append(raw_full)
arr=mne.concatenate_raws(arr)
arr.save("unalteredbutfiltered0.1-40.fif")


In [ ]:


# Run ICA Per Session

def run_ica_per_session(raw_session, session_index):
    """Runs ICA on a single session and removes artifacts."""
    print(f"Running ICA on session {session_index+1}...")

    # Get initial sample count
    n_samples_before = raw_session.n_times
    print(f"Samples before ICA: {n_samples_before}")

    # Run ICA
    ica = mne.preprocessing.ICA(n_components=32, random_state=97, max_iter=1000, method='fastica')
    ica.fit(raw_session)
    
    # Identify & Remove Artifacts
    eog_indices, _ = ica.find_bads_eog(raw_session, ch_name=["Fp1", "Fp2"])
    ica.exclude = eog_indices

    # Apply ICA Correction
    raw_ica = ica.apply(raw_session)

    # Get final sample count
    n_samples_after = raw_ica.n_times
    print(f"Samples after ICA: {n_samples_after}")

    # Check if the sample count changed
    if n_samples_before != n_samples_after:
        print(f"⚠️ WARNING: Sample count changed for session {session_index+1}!")

    return raw_ica

# ------------------------------- #
# Process All Sessions
# ------------------------------- #

def process_eeg_sessions(raw_filename, cleaned_filename, split_indices):
    """Loads raw EEG, runs ICA per session, and saves cleaned EEG as a single FIF file."""

    # Load raw EEG from .fif file
    raw = mne.io.read_raw_fif(raw_filename, preload=True)

    # Prepare session split points
    starts = [0] + list(split_indices)  
    ends = list(split_indices) + [raw.n_times]  
    print("Calculated starts:", starts)
    print("Calculated ends:", ends)

    # Start ICA Cleaning Process
    print("Starting ICA cleaning process...")
    cleaned_data = []

    for i, (start_idx, end_idx) in tqdm(enumerate(zip(starts, ends)), total=len(starts)):
        print(f"\nProcessing session {i+1}/{len(starts)}... (Samples {start_idx} to {end_idx})")

        if start_idx >= end_idx:
            print(f"Skipping session {i+1} (Invalid range: start={start_idx}, end={end_idx})")
            continue  

        # Convert sample indices to time
        tmin = start_idx / raw.info["sfreq"]
        
        # For tmax, use the actual time of the last sample in the segment.
        # This prevents floating point errors when end_idx is raw.n_times.
        if end_idx == raw.n_times:
            tmax = raw.times[-1] 
        else:
            tmax = (end_idx - 1) / raw.info["sfreq"] # end_idx is exclusive, so -1 to get the last sample within range
            
        # Select session data
        # Use include_tmax=True by default which is fine when tmax is adjusted to raw.times[-1]
        raw_session = raw.copy().crop(tmin=tmin, tmax=tmax)

        # Run ICA
        raw_ica = run_ica_per_session(raw_session, i)

        # Append cleaned data
        cleaned_data.append(raw_ica)

    # Merge all cleaned sessions into a single MNE object
    raw_cleaned = mne.concatenate_raws(cleaned_data)

    # Save the final cleaned EEG as a single `.fif` file
    print(f"Saving cleaned EEG to {cleaned_filename}...")
    raw_cleaned.save(cleaned_filename, overwrite=True)
    print(f"ICA cleaning process completed successfully! Cleaned data saved as {cleaned_filename}")

# ------------------------------- #
# Step 4: Run Everything
# ------------------------------- #

# Define filenames
raw_filename = "cleaned_noica_5-12.fif"
cleaned_filename = "cleaned_withica.5-12.fif"


process_eeg_sessions(raw_filename, cleaned_filename, split_indices)

NameError: name 'split_indices' is not defined